In [8]:
! pip install pandas
! pip install twikit
! pip install selenium
! pip install BeautifulSoup


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached BeautifulSoup-3.2.2.tar.gz (32 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [25 lines of output]
      Traceback (most recent call last):
        File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\Public\Documents\ESTsoft\CreatorTemp\pip-build-env-23ksf6w3\overlay\Lib\s

In [ ]:
import asyncio
import csv
import random
import os
from twikit import Client

# --- 설정 및 경로 ---
COOKIES_FILE = 'x_cookies.json'
CSV_FILE = 'tweets_data.csv'
# 여러 키워드를 검색할 때는 "키워드1 OR 키워드2" 형식을 사용해야 합니다.
KEYWORD_LIST = ["전쟁", "분쟁", "위성", "군사", "병력"]
SEARCH_QUERY = " OR ".join(KEYWORD_LIST) 
TARGET_LIMIT = 10

FREE_PROXIES = [
    'http://43.152.113.120:2315',
    'http://111.225.153.131:10087'
]

async def login_setup(client):
    """쿠키를 설정하거나 파일에서 로드하는 함수"""
    auth_token = "7b9a161d5e53b76970ffc0d255730ff91404b1a2"
    ct0 = "699bdecd102d9feff5c6b9f82ea274e05e940db24b9b90f42f7101f54327dde5b082dc869eb879ca4058b6759bb6db957af9d0e42fb2f616580c38a7ff286be6354a79a08cf3e997dae642823d2b8076"
    
    # 쿠키 파일이 있으면 로드, 없으면 새로 설정
    if os.path.exists(COOKIES_FILE):
        client.load_cookies(COOKIES_FILE)
        print("기존 쿠키 파일을 로드했습니다.")
    else:
        client.set_cookies({
            'auth_token': auth_token,
            'ct0': ct0
        })
        client.save_cookies(COOKIES_FILE)
        print("새로운 쿠키를 설정하고 저장했습니다.")

async def main():
    # 1. 클라이언트 초기화
    proxy = random.choice(FREE_PROXIES)
    client = Client('en-US', proxy=proxy)

    # 2. 로그인 (await 필수)
    await login_setup(client)

    # 3. CSV 파일 준비
    file_exists = os.path.isfile(CSV_FILE)
    f = open(CSV_FILE, 'a', encoding='utf-8-sig', newline='')
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['ID', 'Username', 'Text', 'Created_At', 'Retweets', 'Favorites'])

    # 4. 데이터 수집
    print(f"'{SEARCH_QUERY}' 검색 시작 (목표: {TARGET_LIMIT}개)...")
    count = 0
    
    try:
        # 첫 검색 결과 가져오기
        tweets = await client.search_tweet(SEARCH_QUERY, product='Latest')

        while count < TARGET_LIMIT:
            if not tweets:
                print("더 이상 트윗이 없습니다.")
                break

            for tweet in tweets:
                if count >= TARGET_LIMIT:
                    break
                
                writer.writerow([
                    tweet.id, 
                    tweet.user.name, 
                    tweet.text.replace('\n', ' '), 
                    tweet.created_at, 
                    tweet.retweet_count, 
                    tweet.favorite_count
                ])
                count += 1
                print(f"[{count}/{TARGET_LIMIT}] 수집 중: {tweet.user.name}")

            if count < TARGET_LIMIT:
                await asyncio.sleep(random.uniform(3, 5))
                tweets = await tweets.get_next()
            else:
                break

    except Exception as e:
        print(f"수집 중 에러 발생: {e}")
    finally:
        f.close()
        print(f"수집 완료. '{CSV_FILE}' 파일을 확인하세요.")

# --- 실행 부분 ---
# Jupyter Notebook 환경에서는 아래와 같이 실행해야 합니다.
await main()

새로운 쿠키를 설정하고 저장했습니다.
'전쟁 OR 분쟁 OR 위성 OR 군사 OR 병력 OR 대비' 검색 시작 (목표: 10개)...
수집 중 에러 발생: 
수집 완료. 'tweets_data.csv' 파일을 확인하세요.
